In [9]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
import re
import xml.etree.ElementTree as ET
from functools import reduce
import openpyxl
from scipy.optimize import minimize_scalar
import numpy as np
import glob

pd.set_option('display.max_columns', None)

In [10]:
MONETARY_BASE_URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/monetary"

In [11]:
def fetch_monetary_series(id_api, start="20100101", end="20251231", k076="total"):
    params = {
        "start": start,
        "end": end,
        "id_api": id_api,
        "k076": k076,
        "json": ""
    }

    response = requests.get(MONETARY_BASE_URL, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data)

    if df.empty:
        return df

    df["dt"] = pd.to_datetime(df["dt"], format="%Y%m%d")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    df = (
        df[["dt", "id_api", "k076", "freq", "value"]]
        .sort_values("dt")
        .reset_index(drop=True)
    )

    return df

In [12]:
m0_monthly = fetch_monetary_series("M0")
m1_monthly = fetch_monetary_series("M1")
m2_monthly = fetch_monetary_series("M2")

monthly = pd.concat([m0_monthly, m1_monthly, m2_monthly], ignore_index=True)

quarterly = (
    monthly
    .set_index("dt")
    .groupby("id_api")
    .resample("QE")["value"]
    .last()
    .reset_index()
)
quarterly_wide = quarterly.pivot(index="dt", columns="id_api", values="value").reset_index()
quarterly_wide.columns.name = None
quarterly_wide = quarterly_wide.rename(columns={"dt": "quarter_end"})
quarterly_wide['M0'] = quarterly_wide['M0'] * 1e6
quarterly_wide['M1'] = quarterly_wide['M1'] * 1e6
quarterly_wide['M2'] = quarterly_wide['M2'] * 1e6

quarterly_wide["quarter_end"] = pd.to_datetime(quarterly_wide["quarter_end"]).dt.to_period("Q").astype(str)
quarterly_wide = quarterly_wide[quarterly_wide["quarter_end"] < "2025Q1"].reset_index(drop=True)
quarterly_wide.to_csv("data/monetary_aggregates.csv", index=False)

quarterly_wide

,quarter_end,M0,M1,M2
0,2010Q1,1.539822e+11,2.275528e+11,4.780519e+11
1,2010Q2,1.621290e+11,2.492082e+11,5.199940e+11
2,2010Q3,1.751030e+11,2.713026e+11,5.553273e+11
3,2010Q4,1.733323e+11,2.763735e+11,5.726598e+11
4,2011Q1,1.775604e+11,2.867065e+11,6.030967e+11
5,2011Q2,1.846835e+11,3.006278e+11,6.327842e+11
6,2011Q3,1.939773e+11,3.111016e+11,6.610108e+11
7,2011Q4,1.841643e+11,2.948371e+11,6.502994e+11
8,2012Q1,1.864667e+11,3.000397e+11,6.762243e+11
9,2012Q2,1.948008e+11,3.136343e+11,6.948994e+11
